In [ ]:
!pip install unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

# --- Load the Model ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, "kaushik2202/diet-advisor-qwen-qlora")
model.to("cuda")
FastLanguageModel.for_inference(model)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-8B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


adapter_config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/349M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
        (layers): ModuleList(
          (0-2): 3 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
         

In [ ]:
# --- Encapsule ---
def get_diet_advice(user_prompt):

    formatted_prompt = f"<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens=300,
        temperature=0.7,
        use_cache=True
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split("assistant\n")[-1]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/cluster_2_Metabolic_Risk_Obesity.csv'
df = pd.read_csv(file_path)

# select 5 diverse rows
sample_users = df.sample(5, random_state=42)

In [ ]:
def build_prompt(row, style="structured"):

    weight = row["Weight_kg"]
    height = row["Height_cm"]
    bmi = row["BMI"]
    waist = row["Waist_cm"]
    whr = row["Waist_Height_Ratio"]
    sedentary = row["Sedentary_Minutes_Per_Day"]
    sleep = row["Sleep_Hours_Weekdays"]
    sleep_issue = row["Has_Sleep_Issues"]
    diabetes = row["Clinical_Diabetes_Flag"]

    if style == "structured":

        return f"""
Human: I'm seeking dietary advice.

Health Profile:
• Weight: {weight} kg
• Height: {height} cm
• BMI: {bmi}
• Waist circumference: {waist} cm
• Waist-to-height ratio: {whr}

Lifestyle:
• Sedentary minutes per day: {sedentary}
• Sleep per night: {sleep} hours
• Sleep issues: {sleep_issue}

Medical:
• Diabetes diagnosis: {diabetes}

Goal:
Improve metabolic health and reduce obesity risk.

Request:
1. Evaluate my metabolic risk.
2. Recommend daily calorie and macro targets.
3. Provide a 2-day healthy meal plan.
"""

    elif style == "concise":

        return f"""
Human: I weigh {weight}kg and I'm {height}cm tall with BMI {bmi}.
I sit about {sedentary} minutes per day and sleep {sleep} hours.

I may have sleep issues: {sleep_issue}
Diabetes diagnosis: {diabetes}

What diet should I follow to improve my metabolic health?
"""

    elif style == "expert":

        return f"""
Human: Please act as a clinical nutritionist.

Patient metrics:
Weight: {weight} kg
Height: {height} cm
BMI: {bmi}
Waist circumference: {waist} cm
Sedentary time: {sedentary} minutes/day
Sleep: {sleep} hours/night
Sleep issues: {sleep_issue}
Diabetes: {diabetes}

Tasks:
1. Assess obesity and metabolic risk
2. Recommend calorie target
3. Suggest macro ratio (protein/carbs/fats)
4. Provide a structured meal plan table
"""

    elif style == "behavioral":

        return f"""
Human: I'm trying to improve my lifestyle.

My stats:
Weight: {weight}kg
Height: {height}cm
BMI: {bmi}

Lifestyle:
Sedentary minutes/day: {sedentary}
Sleep: {sleep} hours
Sleep issues: {sleep_issue}

Health:
Diabetes diagnosis: {diabetes}

Can you suggest realistic dietary changes and a simple meal plan?
"""

    elif style == "step_by_step":

        return f"""
Human: Analyze my health data step by step.

Step 1: Evaluate obesity risk
Step 2: Estimate healthy calorie intake
Step 3: Suggest macro targets
Step 4: Provide a 2-day meal plan

My data:
Weight: {weight} kg
Height: {height} cm
BMI: {bmi}
Waist: {waist} cm
Sedentary minutes/day: {sedentary}
Sleep: {sleep}
Sleep issues: {sleep_issue}
Diabetes: {diabetes}
"""

In [ ]:
styles = ["structured", "concise", "expert", "behavioral", "step_by_step"]

for i, row in sample_users.iterrows():

    print("\n" + "="*80)
    print("USER:", row["User_ID"])
    print("="*80)

    for style in styles:

        prompt = build_prompt(row, style)

        response = get_diet_advice(prompt)

        print("\n--- PROMPT STYLE:", style, "---")
        print(response)


USER: 112123


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 


--- PROMPT STYLE: structured ---
Assistant: I'll create a personalized dietary plan based on your health profile and current eating patterns.

## 🍽️ Age-Specific Eating Tips (59 years old)

## 📋 Sample Daily Menu

**Breakfast:** Black bean tacos
**Lunch:** Grilled fish with vegetables
**Dinner:** Quinoa stuffed peppers

## 🔄 Improving Your Current Eating Habits

• Continue building on your current healthy eating foundation

## Age-Specific Health Recommendations

• Balance convenience with nutrition quality
• Include calcium-rich foods for bone health

**Remember:** These recommendations are tailored to your health profile and eating patterns. Consult with a registered dietitian for detailed meal planning.

--- PROMPT STYLE: concise ---
Assistant: I'll create a personalized dietary plan based on the user's health profile and current eating patterns.

## 🍽️ Mexican Recommendations for Metabolic Health

## 📋 Sample Daily Menu

**Breakfast:** Black bean tacos
**Lunch:** Grilled fish with

**Model weaknesses:**

1. Lack of personalization

    Even though the prompts include weight, BMI, sedentary minutes and diabetes flag, the responses do not reference them at all.

    e.g. “Continue building on your current healthy eating foundation.” - This is generic advice.


2. Hallucinated age

    The model repeatedly invents random ages even though age was never provided in the dataset. e.g. Age-Specific Eating Tips (65 years old)


3.  Mode collapse (same menu everywhere)

    Nearly every output contains:

    Breakfast: Greek salad with grilled chicken
    Lunch: Black bean tacos
    Dinner: Grilled fish with vegetables

    This means the LoRA likely memorized a few meal templates.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# Load Model and tokenizer
device='cuda' if torch.cuda.is_available() else 'cpu'

model_path = "Qwen/Qwen2.5-0.5B"

# One training model and one reference model
model = AutoModelForCausalLM.from_pretrained(model_path)
model_ref = AutoModelForCausalLM.from_pretrained(model_path)
model.train()
model_ref.eval()
model.to(device)
model_ref.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load Data
dpo_train_data = [

{
"prompt": """
Food image analysis:

Calories: 620 kcal
Protein: 22 g
Carbs: 85 g
Fat: 18 g

User profile:
Goal: Weight loss
Weight: 80 kg
Height: 175 cm
Age: 30
Activity level: moderately active

Provide dietary feedback and suggestions.
""",

"chosen": """
This meal is relatively high in carbohydrates and moderate in calories.
For a weight loss goal, increasing protein and slightly reducing carbs can improve satiety.

Suggestions:
- Add a lean protein source such as grilled chicken, tofu, or Greek yogurt.
- Replace part of the refined carbs with vegetables or whole grains.
- Aim for a macro balance closer to 35% protein, 25% fat, and 40% carbs for weight loss.

For your next meal, prioritize:
- Lean protein (fish, chicken breast, lentils)
- High-fiber vegetables
- Moderate complex carbs like quinoa or brown rice.
""",

"rejected": """
This looks like a normal meal. Just keep eating healthy foods and exercise regularly.
"""
},

{
"prompt": """
Food image analysis:

Calories: 450 kcal
Protein: 12 g
Carbs: 60 g
Fat: 15 g

User profile:
Goal: Muscle gain
Weight: 70 kg
Height: 170 cm
Age: 25
Activity level: very active

Give recommendations.
""",

"chosen": """
For muscle gain, your protein intake in this meal is too low.

At 70 kg body weight, a good target is around 1.6–2.2 g protein per kg daily.

Suggestions:
- Increase protein by adding eggs, chicken breast, tofu, or protein shakes.
- Maintain carbohydrates for training energy.
- Include healthy fats like nuts or olive oil.

For your next meal, aim for:
- 30–40 g protein
- balanced carbs to fuel workouts
- healthy fats in moderation.
""",

"rejected": """
This meal contains carbs, protein, and fat. It is balanced and should help you gain muscle.
"""
},

{
"prompt": """
Food image analysis:

Calories: 700 kcal
Protein: 25 g
Carbs: 95 g
Fat: 20 g

User profile:
Goal: Maintain weight
Weight: 65 kg
Height: 168 cm
Age: 29
Activity level: lightly active
""",

"chosen": """
This meal is relatively high in carbohydrates but acceptable for maintenance.

To maintain stable energy levels:
- Include more fiber sources such as vegetables or legumes.
- Balance the plate with additional lean protein.

A good maintenance macro balance is around:
30% protein / 30% fat / 40% carbohydrates.

Future meal ideas:
- grilled salmon with quinoa and vegetables
- tofu stir-fry with brown rice
""",

"rejected": """
Your meal has food in it. Try to eat healthy meals every day.
"""
}

]

# Optimizer
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)

# Data Preprocess
def preprocess(tokenizer, batch_messages):
    input_list = []
    target_list = []

    im_start = tokenizer('<|im_start|>').input_ids
    im_end = tokenizer('<|im_end|>').input_ids
    newline = tokenizer('\n').input_ids
    pad = tokenizer('<|endoftext|>').input_ids
    ignore = [-100]

    for group in batch_messages:

        input_ids = []
        target_ids = []
        for msg in group:
            role = tokenizer(msg['role']).input_ids
            content = tokenizer(msg['content']).input_ids
            if msg['role'] in ['system', 'user']:
                ignore_parts = role + newline + content
                input_ids += im_start + ignore_parts + im_end + newline
                target_ids += im_start + ignore * len(ignore_parts) + im_end + newline
            else:
                ignore_parts = role + newline
                input_ids += im_start + ignore_parts + content + im_end + newline
                target_ids += im_start + ignore * len(ignore_parts) + content + im_end + newline
        input_list.append(input_ids)
        target_list.append(target_ids)

    # padding
    max_len = max([len(ids) for ids in input_list])
    for input_ids, target_ids in zip(input_list, target_list):
        input_ids += pad * (max_len - len(input_ids))
        target_ids += ignore * (max_len - len(target_ids))
    batch_input_ids = torch.tensor(input_list, dtype=torch.long)
    batch_target_ids = torch.tensor(target_list, dtype=torch.long)
    batch_mask = batch_input_ids.ne(pad[0]).type(torch.long)
    return batch_input_ids, batch_target_ids, batch_mask

# preference dataset -> model input
def dpo_to_messages(dpo_pairs):
    chosen_messages=[]
    reject_messages=[]
    for pair in dpo_pairs:
        chosen_messages.append([
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": pair['prompt']},
                {"role": "assistant", "content": pair['chosen']},
            ]
        )
        reject_messages.append([
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": pair['prompt']},
                {"role": "assistant", "content": pair['rejected']},
            ]
        )


    return chosen_messages,reject_messages

# DPOloss computation
def dpo_prob_calc(target_ids, pi_logits, ref_logits):
    pi_probs = torch.log_softmax(pi_logits, dim=-1)
    ref_probs = torch.log_softmax(ref_logits, dim=-1)

    ignore_mask = target_ids != -100
    indexes = target_ids * ignore_mask

    pi_probs_of_target = torch.gather(pi_probs, dim=-1, index=indexes.unsqueeze(-1)).squeeze(
        -1) * ignore_mask
    ref_probs_of_target = torch.gather(ref_probs, dim=-1, index=indexes.unsqueeze(-1)).squeeze(-1) * ignore_mask

    pi_final_prob = pi_probs_of_target.sum(-1) / ignore_mask.sum(-1)
    ref_final_prob = ref_probs_of_target.sum(-1) / ignore_mask.sum(-1)
    return pi_final_prob, ref_final_prob


# DPO loss function https://github.com/huggingface/trl/blob/main/trl/trainer/dpo_trainer.py
def dpo_loss(params):
    ## chosen output
    chosen_target_ids = params['chosen_target_ids'][:, 1:]
    pi_chosen_logits = params['pi_chosen_logits'][:, :-1, :]
    ref_chosen_logits = params['ref_chosen_logits'][:, :-1, :]
    pi_chosen_prob, ref_chosen_prob = dpo_prob_calc(chosen_target_ids, pi_chosen_logits, ref_chosen_logits)

    ## reject output
    reject_target_ids = params['reject_target_ids'][:, 1:]
    pi_reject_logits = params['pi_reject_logits'][:, :-1, :]
    ref_reject_logits = params['ref_reject_logits'][:, :-1, :]
    pi_reject_prob, ref_reject_prob = dpo_prob_calc(reject_target_ids, pi_reject_logits, ref_reject_logits)

    # DPO Loss
    pi_prob_diff = pi_chosen_prob - pi_reject_prob
    ref_prob_diff = ref_chosen_prob - ref_reject_prob
    beta = 0.1
    loss = -torch.nn.functional.logsigmoid(beta * (pi_prob_diff - ref_prob_diff))
    return loss.mean()

print("Start Training...")
iterators=5
vocab=tokenizer.get_vocab()
for i in range(iterators):
    chosen_messages,reject_messages=dpo_to_messages(dpo_train_data)
    chosen_input_ids,chosen_target_ids,chosen_mask=preprocess(tokenizer,chosen_messages)
    reject_input_ids,reject_target_ids,reject_mask=preprocess(tokenizer,reject_messages)
    pi_chosen_logits=model(input_ids=chosen_input_ids.to(device),attention_mask=chosen_mask.to(device)).logits
    pi_reject_logits=model(input_ids=reject_input_ids.to(device),attention_mask=reject_mask.to(device)).logits
    ref_chosen_logits=model_ref(chosen_input_ids.to(device),chosen_mask.to(device)).logits
    ref_reject_logits=model_ref(reject_input_ids.to(device),reject_mask.to(device)).logits
    loss=dpo_loss({
        'chosen_target_ids':chosen_target_ids.to(device),
        'reject_target_ids':reject_target_ids.to(device),
        'pi_chosen_logits':pi_chosen_logits.to(device),
        'pi_reject_logits':pi_reject_logits.to(device),
        'ref_chosen_logits':ref_chosen_logits.to(device),
        'ref_reject_logits':ref_reject_logits.to(device),
    })
    print(loss)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Save Model")
model.save_pretrained("model/trained")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Start Training...
tensor(0.6914, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
tensor(0.3047, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
tensor(0.0986, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
tensor(0.0233, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
tensor(0.0079, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
Save Model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]